# 02 — Feature Engineering with Spectral Indices
**GeoAI Aquaculture Pond Identification Challenge**

We engineer per-timestep spectral indices from awesome-spectral-indices  
(https://github.com/awesome-spectral-indices/awesome-spectral-indices)  
then aggregate across the 12 monthly composites with mean/std/min/max/range.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier

from src.features import prepare_Xy, build_features
from src.preprocessing import replace_missing, load_train, load_test

DATA = pathlib.Path('..') / 'data' / 'raw'
sns.set_theme(style='whitegrid', palette='colorblind')
plt.rcParams['figure.dpi'] = 110

## 1. Build features

In [ ]:
X_train, y_train, X_test = prepare_Xy(DATA / 'Train.csv', DATA / 'Test.csv')
print(f'X_train: {X_train.shape}  |  X_test: {X_test.shape}')
print(f'\nFeature names (first 20):\n', list(X_train.columns[:20]))

## 2. Missing values in features

In [ ]:
nan_pct = X_train.isna().mean() * 100
high_nan = nan_pct[nan_pct > 5].sort_values(ascending=False)
print(f'Features with >5% missing: {len(high_nan)}')
if len(high_nan) > 0:
    high_nan.head(20).plot.bar(figsize=(10, 3), title='% missing > 5%')
    plt.tight_layout(); plt.show()
else:
    print('All features have <5% missing — good!')

## 3. Key water indices — pond vs no-pond

In [ ]:
water_indices = ['ndwi_mean', 'ndwi_std', 'awei_sh_mean', 'awei_nsh_mean']
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
axes = axes.flatten()

for ax, col in zip(axes, water_indices):
    for label, color, name in [(0, '#4C72B0', 'No pond'), (1, '#DD8452', 'Pond')]:
        vals = X_train.loc[y_train == label, col].dropna()
        ax.hist(vals, bins=40, alpha=0.65, color=color, label=name, density=True)
    ax.set_title(col)
    ax.legend(fontsize=8)

plt.suptitle('Water index distributions by class', fontsize=13)
plt.tight_layout(); plt.show()

In [ ]:
veg_indices = ['ndvi_mean', 'ndvi_std', 'bsi_mean', 'ndre_mean']
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
axes = axes.flatten()

for ax, col in zip(axes, veg_indices):
    for label, color, name in [(0, '#4C72B0', 'No pond'), (1, '#DD8452', 'Pond')]:
        vals = X_train.loc[y_train == label, col].dropna()
        ax.hist(vals, bins=40, alpha=0.65, color=color, label=name, density=True)
    ax.set_title(col)
    ax.legend(fontsize=8)

plt.suptitle('Vegetation/soil index distributions by class', fontsize=13)
plt.tight_layout(); plt.show()

## 4. Quick feature importance (Random Forest proxy)

In [ ]:
# Fill NaN with median for RF
X_filled = X_train.fillna(X_train.median())

rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1, class_weight='balanced')
rf.fit(X_filled, y_train)

importance = pd.Series(rf.feature_importances_, index=X_train.columns)
top_features = importance.nlargest(30)

fig, ax = plt.subplots(figsize=(10, 7))
top_features.sort_values().plot.barh(ax=ax, color='steelblue')
ax.set_title('Top-30 features by RF importance')
ax.set_xlabel('Importance')
plt.tight_layout(); plt.show()

In [ ]:
# Group importance by index category
categories = {
    'SAR (VH/VV)': lambda c: c.startswith('vh') or c.startswith('vv') or c.startswith('sar'),
    'NDWI/AWEI (water)': lambda c: 'ndwi' in c or 'awei' in c,
    'NDVI/EVI/SAVI (veg)': lambda c: any(x in c for x in ['ndvi', 'evi', 'savi']),
    'BSI/NDRE/CIG (soil/RE)': lambda c: any(x in c for x in ['bsi', 'ndre', 'cig']),
    'Raw optical bands': lambda c: any(c.startswith(x) for x in ['blue','green','red','nir','swir','re1','re2','re3','nira']),
    'Count / validity': lambda c: 'valid' in c or 'n_opt' in c,
}

cat_importance = {}
for cat, fn in categories.items():
    cat_importance[cat] = importance[[c for c in importance.index if fn(c)]].sum()

cat_series = pd.Series(cat_importance).sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(7, 4))
cat_series.plot.barh(ax=ax, color='coral')
ax.set_title('Feature importance by category')
ax.set_xlabel('Total RF importance')
plt.tight_layout(); plt.show()

## 5. Feature correlation heatmap (top features)

In [ ]:
top20 = top_features.index.tolist()
corr = X_filled[top20].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=False, cmap='RdBu_r', center=0, ax=ax, square=True, linewidths=0.5)
ax.set_title('Top-20 feature correlation matrix')
plt.tight_layout(); plt.show()

## Summary

| Category | Signal |
|---|---|
| NDWI / AWEI | Strongest water discriminators — ponds have high values |
| SAR VH/VV | Smooth water → lower backscatter than land |
| NDVI | Low for open water — useful negative signal |
| BSI | High for bare soil / built-up — helps exclude non-pond |
| Temporal std | Seasonality differs between ponds and crops |
| n_opt_months | Ponds may be in regions with fewer cloudy days |